# 04 Experiment Tracking
Using MLflow to track hyperparameters, metrics, and models.

In [ ]:
import pandas as pd
import mlflow
import sys
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv
import os

# Load environment variables (.env)
load_dotenv('../.env')

# Add src to path
sys.path.append('..')
from src.train import train_xgb_model

print('MLflow ready!')

## 1. Setup MLflow
Set the tracking URI and create/select an experiment.

In [ ]:
tracking_uri = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment('Predictive_Maintenance_RUL')

print(f'Tracking at: {tracking_uri}')

## 2. Load Data

In [ ]:
df = pd.read_parquet('../data/processed/features_train.parquet')
X = df.drop('RUL', axis=1)
y = df['RUL']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Run Multiple Experiments
We will try different hyperparameters and log them all to MLflow.

In [ ]:
experiment_configs = [
    {'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 4, 'name': 'XGB_Small_Fast'},
    {'n_estimators': 1000, 'learning_rate': 0.05, 'max_depth': 6, 'name': 'XGB_Standard'},
    {'n_estimators': 2000, 'learning_rate': 0.01, 'max_depth': 8, 'name': 'XGB_Deep_Slow'}
]

for config in experiment_configs:
    params = {
        'n_estimators': config['n_estimators'],
        'learning_rate': config['learning_rate'],
        'max_depth': config['max_depth'],
        'objective': 'reg:squarederror',
        'random_state': 42
    }
    
    model, metrics = train_xgb_model(
        X_train, y_train, X_val, y_val, 
        params=params, 
        use_gpu=True, 
        run_name=config['name']
    )
    print(f"Finished {config['name']}: RMSE={metrics['rmse']:.4f}")

## 4. View Results
To see the dashboard, run the following in your terminal:
```bash
mlflow ui
```
Then visit `http://localhost:5000` in your browser.